# Cleaning

- ## 1 - Tests
- ## 2 - Merging of all files
- ## 3 - Understanding
- ## 4 - Duplicates
- ## 5 - Fill
- ## 6 - Outliers
- ## 7 - End

## 1 - Tests

In [2]:
import numpy as np
import pandas as pd
import os

directory = os.fsencode("synop")
#for file in os.listdir(directory):
#    filename = os.fsdecode(file)
 #   df = pd.read_csv("synop/{}".format(filename),sep=";")
  #  print("{} contain {} columns".format(filename,df.columns.size)) 
   # print("---------")

In [ ]:


filename = "synop/synop_1996.csv"

df = pd.read_csv(filename,sep=";")
df.info()

In [ ]:
# On supprime les doublons
df_net=df.drop_duplicates()
df_net.info()

In [ ]:
# On supprime les doublons
df.duplicated().sum()
#df_net.isnull().sum()


In [ ]:
df.info()

In [ ]:
df_del=df.drop_duplicates()
df_del.info()

## 2 - Merging all files

In [3]:
from pathlib import Path

files = sorted(Path("synop").glob("*.csv"))

dfs = []

for file in files:
    df = pd.read_csv(file, sep=";")
    df["source_file"] = file.name
    dfs.append(df)

synop = pd.concat(dfs, ignore_index=True)

C:\Users\MV_pe\AppData\Local\Temp\ipykernel_2568\2636828541.py:8: DtypeWarning: Columns (0: reference_time, 1: insert_time) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=";")


## 3 - Understanding

In [ ]:
synop.shape


In [ ]:
synop.info()

In [ ]:
pd.set_option('display.max_rows',None)
synop.isna().sum().sort_values(ascending=False)


## 4 - Cleaning

### Duplicates

In [4]:
float_cols = ["t", "u", "ff", "dd", "pmer", "rr3"]

synop[float_cols] = synop[float_cols].astype("float32")

In [ ]:
synop.duplicated().sum()

In [5]:
synop=synop.drop_duplicates()

In [ ]:
synop.duplicated().sum()

In [ ]:
synop.duplicated(
    subset=["geo_id_wmo", "validity_time"]
).sum()

In [6]:
synop = synop.drop_duplicates(
    subset=["geo_id_wmo", "validity_time"],
    keep="last"
)

In [ ]:
synop.duplicated(
    subset=["geo_id_wmo", "validity_time"]
).sum()

### Filling

In [ ]:
synop["rr3"].isna().sum()

In [7]:
cols = ["t", "u", "ff", "pmer"]

synop = synop.sort_values(["geo_id_wmo", "validity_time"])

synop[cols] = (
    synop.groupby("geo_id_wmo")[cols]
      .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)




In [8]:
for col in ["t", "u", "ff", "pmer"]:
    synop[col] = synop[col].fillna(
        synop.groupby("geo_id_wmo")[col].transform("mean")
    )
for col in ["t", "u", "ff", "pmer"]:
    synop[col] = synop[col].fillna(synop[col].mean())

In [ ]:
synop["u"].isna().sum()

In [9]:
synop["dd"] = (
    synop.groupby("geo_id_wmo")["dd"]
      .transform(lambda x: x.ffill().bfill())
)

In [ ]:
synop[["t", "u", "ff", "pmer"]].isna().sum()

### Outliers

In [ ]:
cols = ["t", "u", "ff", "pmer"]

outliers_iqr = {}

for col in cols:
    Q1 = synop[col].quantile(0.25)
    Q3 = synop[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers_iqr[col] = synop[(synop[col] < lower) | (synop[col] > upper)][col]
    print(f"{col}: {len(outliers_iqr[col])} outliers detected")

In [ ]:
print(synop.columns)

In [10]:
cols = ["t", "u", "ff", "pmer"]

mask = pd.Series(False,index=synop.index)

for col in cols:
    Q1 = synop.groupby("geo_id_wmo")[col].transform(
        lambda x: x.quantile(0.25)
    )

    Q3 = synop.groupby("geo_id_wmo")[col].transform(
        lambda x: x.quantile(0.75)
    )

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask|= ~synop[col].between(lower,upper)
    synop[col + "_outlier"] = ~synop[col].between(lower, upper)

synop_clean = synop[~mask]

In [12]:
cols = ["t", "u", "ff", "pmer"]

for col in cols:
    bounds = synop_clean.groupby("geo_id_wmo")[col].agg(
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    bounds["IQR"] = bounds["Q3"] - bounds["Q1"]
    bounds["lower"] = bounds["Q1"] - 1.5*bounds["IQR"]
    bounds["upper"] = bounds["Q3"] + 1.5*bounds["IQR"]

    
    df = synop_clean.merge(bounds[["lower","upper"]], left_on="geo_id_wmo", right_index=True)
    
    synop_clean[col] = df[col].clip(lower=df["lower"], upper=df["upper"])

In [13]:
cols = ["t", "u", "ff", "pmer"]
outliers_iqr = {}

for col in cols:
    
    bounds = synop_clean.groupby("geo_id_wmo")[col].agg(
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    bounds["IQR"] = bounds["Q3"] - bounds["Q1"]
    bounds["lower"] = bounds["Q1"] - 1.5*bounds["IQR"]
    bounds["upper"] = bounds["Q3"] + 1.5*bounds["IQR"]

    
    df = synop_clean.merge(bounds[["lower","upper"]], left_on="geo_id_wmo", right_index=True)

   
    mask = (df[col] < df["lower"]) | (df[col] > df["upper"])
    outliers_iqr[col] = df[mask][col]
    print(f"{col}: {len(outliers_iqr[col])} outliers detected")

t: 0 outliers detected
u: 0 outliers detected
ff: 0 outliers detected
pmer: 0 outliers detected


In [ ]:
print(synop_clean[["t","u","ff","pmer"]].describe())

In [ ]:
print(synop_clean.columns)

### End

In [14]:
final_synop = synop_clean[
    [
        "geo_id_wmo",
        "geo_id_wigos",
        "name",
        "lat",
        "lon",
        "validity_time",
        "t",
        "u",
        "ff",
        "dd",
        "pmer",
        "rr3"
    ]
]

In [15]:
final_synop = final_synop.copy()

final_synop["rr3"] = (
    pd.to_numeric(final_synop["rr3"], errors="coerce")
    .fillna(0)
)
final_synop.to_csv("clean_synop.csv",index=False)

In [ ]:
final_synop.shape
